In [2]:
import os
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
from collections import Counter
from tqdm import tqdm
from openai import OpenAI

##############################################
# OpenAI Client
##############################################

client = OpenAI(api_key=os.getenv("OPEN_API_KEY"))

##############################################
# 1 Load Documents (RAW ONLY)
##############################################

def load_docs(folder_path):

    documents = []

    for file in os.listdir(folder_path):

        if file.endswith((".puml", ".plantuml")):

            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:

                content = f.read()

                documents.append({
                    "id": file,
                    "label": file.split('-')[0].split('_')[0],
                    "content": content
                })

    return documents


##############################################
# 2 Embedding Model
##############################################

class EmbeddingModel:

    def __init__(self, model_name="BAAI/bge-base-en"):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = SentenceTransformer(model_name).to(self.device)

    def encode(self, texts, batch_size=16):

        return self.model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )


##############################################
# 3 Vector Store
##############################################

class VectorStore:

    def __init__(self, embedding_dim):

        self.index = faiss.IndexFlatIP(embedding_dim)
        self.metadata = []

    def add_embeddings(self, embeddings, metadatas):

        self.index.add(embeddings)
        self.metadata.extend(metadatas)

    def search(self, query_embedding, top_k=5):

        distances, indices = self.index.search(query_embedding, top_k)

        results = []

        for i, idx in enumerate(indices[0]):

            if idx != -1:

                res = self.metadata[idx].copy()
                res["score"] = distances[0][i]
                results.append(res)

        return results


##############################################
# 4 LLM Reasoning (Optional)
##############################################

def llm_reasoning(query_raw, retrieved_docs):

    examples = "\n\n".join(
        [f"Example ({doc['id']}):\n{doc['content']}" for doc in retrieved_docs]
    )

    prompt = f"""

Identify only the design pattern Singleton used in the following PlantUML diagram. 
If no Singleton is found, respond with "nons".

QUERY:
{query_raw}

SIMILAR EXAMPLES:
{examples}


Answer ONLY with the pattern name.
"""

    response = client.chat.completions.create(
        model="gpt-5.1",
        messages=[
            {"role": "system", "content": "You are an expert in GoF software design patterns."},
            {"role": "user", "content": prompt}
        ],
    )

    return response.choices[0].message.content.strip()


##############################################
# 5 Build Index
##############################################

def build_rag_index(folder_path):

    print("Loading documents...")

    docs = load_docs(folder_path)

    texts = [doc["content"] for doc in docs]

    embedder = EmbeddingModel()

    embeddings = embedder.encode(texts)

    vector_store = VectorStore(embedding_dim=embeddings.shape[1])

    vector_store.add_embeddings(np.array(embeddings), docs)

    return vector_store, embedder


##############################################
# 6 Pattern Detection
##############################################

def detect_pattern(test_doc, vector_store, embedder, top_k=5, use_llm=True):

    query_embedding = embedder.encode([test_doc["content"]])

    results = vector_store.search(query_embedding, top_k=top_k)

    # Voting
    votes = {}

    for r in results:
        label = r["label"]
        votes[label] = votes.get(label, 0) + r["score"]

    predicted_label = max(votes, key=votes.get) if votes else "Unknown"

    if use_llm:
        llm_prediction = llm_reasoning(test_doc["content"], results)
        return llm_prediction, [r["id"] for r in results]

    return predicted_label, [r["id"] for r in results]


##############################################
# 7 Evaluation
##############################################

def evaluate_detection(test_folder, vector_store, embedder, top_k=5):

    test_docs = load_docs(test_folder)

    correct_counts = Counter()
    total_counts = Counter()

    for doc in tqdm(test_docs):

        true_label = doc["label"]
        total_counts[true_label] += 1

        predicted_label, neighbors = detect_pattern(
            doc,
            vector_store,
            embedder,
            top_k=top_k
        )

        if predicted_label.lower() == true_label.lower():
            correct_counts[true_label] += 1

        print(f"File: {doc['id']} | True: {true_label} | Pred: {predicted_label}")

    per_pattern_accuracy = {}

    for label in total_counts:
        per_pattern_accuracy[label] = correct_counts[label] / total_counts[label]

    overall_accuracy = sum(correct_counts.values()) / sum(total_counts.values())

    print("\nPer-Pattern Accuracy:")
    for label, acc in per_pattern_accuracy.items():
        print(f"{label}: {acc:.4f}")

    print(f"\nOverall Accuracy: {overall_accuracy:.4f}")

    return overall_accuracy, per_pattern_accuracy

In [3]:
# ===============================
# 0. Set your OpenAI API Key
# ===============================
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")  # Or set it directly: openai.api_key = "YOUR_KEY"

# ===============================
# 1. Build the RAG index
# ===============================
data_folder = "PMART_PUML/knowledge_base"
vector_store, embedder = build_rag_index(data_folder)



# ===============================
# 2. Evaluate accuracy on a test folder
# ===============================
test_folder = "PMART_PUML/puml_test/"
overall_acc, per_pattern_acc = evaluate_detection(
    test_folder,
    vector_store,
    embedder,
    top_k=5
)

Loading documents...


  1%|          | 1/93 [00:10<16:27, 10.74s/it]

File: nons_56.puml | True: nons | Pred: nons


  2%|▏         | 2/93 [00:17<12:50,  8.47s/it]

File: singleton_21.puml | True: singleton | Pred: Singleton


...

File: nons_19.puml | True: nons | Pred: nons


 99%|█████████▉| 92/93 [15:53<00:10, 10.81s/it]

File: singleton_16.puml | True: singleton | Pred: nons


100%|██████████| 93/93 [16:02<00:00, 10.35s/it]

File: nons_34.puml | True: nons | Pred: Singleton

Per-Pattern Accuracy:
nons: 0.8235
singleton: 0.9200

Overall Accuracy: 0.8495


In [ ]:
import os
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
from collections import Counter
from tqdm import tqdm
from openai import OpenAI

##############################################
# OpenAI Client
##############################################

client = OpenAI(api_key=os.getenv("OPEN_API_KEY"))

##############################################
# 1 Load Documents (RAW ONLY)
##############################################

def load_docs(folder_path):

    documents = []

    for file in os.listdir(folder_path):

        if file.endswith((".puml", ".plantuml")):

            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:

                content = f.read()

                documents.append({
                    "id": file,
                    "label": file.split('-')[0].split('_')[0],
                    "content": content
                })

    return documents


##############################################
# 2 Embedding Model
##############################################

class EmbeddingModel:

    def __init__(self, model_name="BAAI/bge-base-en"):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = SentenceTransformer(model_name).to(self.device)

    def encode(self, texts, batch_size=16):

        return self.model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )


##############################################
# 3 Vector Store
##############################################

class VectorStore:

    def __init__(self, embedding_dim):

        self.index = faiss.IndexFlatIP(embedding_dim)
        self.metadata = []

    def add_embeddings(self, embeddings, metadatas):

        self.index.add(embeddings)
        self.metadata.extend(metadatas)

    def search(self, query_embedding, top_k=5):

        distances, indices = self.index.search(query_embedding, top_k)

        results = []

        for i, idx in enumerate(indices[0]):

            if idx != -1:

                res = self.metadata[idx].copy()
                res["score"] = distances[0][i]
                results.append(res)

        return results


##############################################
# 4 LLM Reasoning (Optional)
##############################################

def llm_reasoning(query_raw, retrieved_docs):

    examples = "\n\n".join(
        [f"Example ({doc['id']}):\n{doc['content']}" for doc in retrieved_docs]
    )

    prompt = f"""

Identify only the design pattern Singleton used in the following PlantUML diagram. 
If no Singleton is found, respond with "nons".

QUERY:
{query_raw}

SIMILAR EXAMPLES:
{examples}


Answer ONLY with the pattern name.
"""

    response = client.chat.completions.create(
        model="gpt-5.1",
        messages=[
            {"role": "system", "content": "You are an expert in GoF software design patterns."},
            {"role": "user", "content": prompt}
        ],
    )

    return response.choices[0].message.content.strip()


##############################################
# 5 Build Index
##############################################

def build_rag_index(folder_path):

    print("Loading documents...")

    docs = load_docs(folder_path)

    texts = [doc["content"] for doc in docs]

    embedder = EmbeddingModel()

    embeddings = embedder.encode(texts)

    vector_store = VectorStore(embedding_dim=embeddings.shape[1])

    vector_store.add_embeddings(np.array(embeddings), docs)

    return vector_store, embedder


##############################################
# 6 Pattern Detection
##############################################

def detect_pattern(test_doc, vector_store, embedder, top_k=5, use_llm=True):

    query_embedding = embedder.encode([test_doc["content"]])

    results = vector_store.search(query_embedding, top_k=top_k)

    # Voting
    votes = {}

    for r in results:
        label = r["label"]
        votes[label] = votes.get(label, 0) + r["score"]

    predicted_label = max(votes, key=votes.get) if votes else "Unknown"

    if use_llm:
        llm_prediction = llm_reasoning(test_doc["content"], results)
        return llm_prediction, [r["id"] for r in results]

    return predicted_label, [r["id"] for r in results]


##############################################
# 7 Evaluation
##############################################

def evaluate_detection(test_folder, vector_store, embedder, top_k=5):

    test_docs = load_docs(test_folder)

    correct_counts = Counter()
    total_counts = Counter()

    for doc in tqdm(test_docs):

        true_label = doc["label"]
        total_counts[true_label] += 1

        predicted_label, neighbors = detect_pattern(
            doc,
            vector_store,
            embedder,
            top_k=top_k
        )

        if predicted_label.lower() == true_label.lower():
            correct_counts[true_label] += 1

        print(f"File: {doc['id']} | True: {true_label} | Pred: {predicted_label}")

    per_pattern_accuracy = {}

    for label in total_counts:
        per_pattern_accuracy[label] = correct_counts[label] / total_counts[label]

    overall_accuracy = sum(correct_counts.values()) / sum(total_counts.values())

    print("\nPer-Pattern Accuracy:")
    for label, acc in per_pattern_accuracy.items():
        print(f"{label}: {acc:.4f}")

    print(f"\nOverall Accuracy: {overall_accuracy:.4f}")

    return overall_accuracy, per_pattern_accuracy